# 07 Stretch: pitch control

Spearman (2018) pitch control at the intended target, parameters from Laurie Shaw's Friends of Tracking tutorial. Code: `features.pitch_control`. Kept only if it beats the headline model on held-out log loss (it doesn't).

In [1]:
import pandas as pd
from passmodel.load import REPO_ROOT

RESULTS = REPO_ROOT / "results"
f = pd.read_parquet(REPO_ROOT / "data" / "processed" / "features.parquet")
f = f[(f["fold"] == f["match_id"]) & ~f["is_set_piece"]]  # each pass as its own test fold saw it

## Completion rises steadily with pitch control at the target

In [2]:
f.groupby(pd.qcut(f["target_pitch_control"], 5), observed=True)["completed"].agg(completion="mean", passes="size").round(3)

,completion,passes
target_pitch_control,,
"(0.105, 0.734]",0.691,669
"(0.734, 0.907]",0.740,669
"(0.907, 0.975]",0.784,668
"(0.975, 0.996]",0.825,669
"(0.996, 1.0]",0.903,669


## ...but it mostly repeats 'space around the target', which the model already has

In [3]:
f[["target_pitch_control", "target_nearest_opp", "lane_min_opp_dist", "passer_nearest_opp"]].corr(method="spearman").round(2)

,target_pitch_control,target_nearest_opp,lane_min_opp_dist,passer_nearest_opp
target_pitch_control,1.00,0.90,0.44,0.26
target_nearest_opp,0.90,1.00,0.47,0.29
lane_min_opp_dist,0.44,0.47,1.00,0.41
passer_nearest_opp,0.26,0.29,0.41,1.00


## Held-out log loss per fold

In [4]:
m = pd.read_csv(RESULTS / "metrics_per_fold.csv")
m.pivot(index="model", columns="fold", values="log_loss")

fold,1,2,3,mean
model,,,,
constant,0.5710,0.4800,0.5111,0.5207
logistic (pass length),0.5664,0.4790,0.5093,0.5183
logistic (pitch control),0.5618,0.4798,0.4949,0.5121
"xgboost (inner-CV tuned, sensitivity)",0.5105,0.4216,0.4462,0.4594
xgboost (single target + pitch control),0.5160,0.4213,0.4604,0.4659
xgboost (single target),0.5201,0.4186,0.4541,0.4643
xgboost (target mixture),0.5037,0.4118,0.4504,0.4553


## Paired bootstrap: (headline + pitch control) minus headline. Negative = pitch control helps

In [5]:
ci = pd.read_csv(RESULTS / "stretch_pitch_control_ci.csv")
ci[ci["metric"] == "delta_log_loss"]

,model,fold,metric,estimate,ci_low,ci_high
3,xgboost (single target + pitch control),1,delta_log_loss,-0.0041,-0.0113,0.0036
7,xgboost (single target + pitch control),2,delta_log_loss,0.0027,-0.0022,0.0082
11,xgboost (single target + pitch control),3,delta_log_loss,0.0062,-0.0003,0.0130
